# Ontology Design for Knowledge Graphs

An **ontology** is a *formal, explicit specification of a shared conceptualization of a domain*. It defines:
- **Classes** (types of entities)
- **Properties** (attributes and relations)
- **Constraints** (domains, ranges, cardinalities)

**Ontology vs. Knowledge Graph:**
- Ontology = the *schema* (what kinds of things and relations are allowed)
- Knowledge Graph = ontology + *data* (concrete facts instantiating that schema)

In agentic systems, ontologies provide a **shared vocabulary and type system** that enables consistent retrieval, reasoning, and tool interoperability.

In [ ]:
%pip install rdflib networkx matplotlib -q

In [ ]:
from rdflib import Graph, Namespace, Literal, URIRef, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Building a Simple Ontology: E-Commerce Domain

We'll build an ontology step-by-step using **rdflib** (Python's RDF library).

Our e-commerce domain has:
- **Classes**: Product, Customer, Order, Category
- **Subclasses**: ElectronicDevice (subclass of Product), Laptop (subclass of ElectronicDevice)
- **Properties**: placesOrder, containsProduct, hasPrice, belongsToCategory

In [ ]:
# Create RDF graph and namespace
g = Graph()
EX = Namespace("http://example.org/ecommerce/")
g.bind("ex", EX)
g.bind("owl", OWL)

# --- Define Classes ---
classes = ["Product", "Customer", "Order", "Category", "ElectronicDevice", "Laptop", "Apparel"]
for cls in classes:
    g.add((EX[cls], RDF.type, OWL.Class))
    g.add((EX[cls], RDFS.label, Literal(cls)))

# --- Subclass hierarchy ---
g.add((EX.ElectronicDevice, RDFS.subClassOf, EX.Product))
g.add((EX.Laptop, RDFS.subClassOf, EX.ElectronicDevice))
g.add((EX.Apparel, RDFS.subClassOf, EX.Product))

# --- Define Object Properties (links between entities) ---
# placesOrder: Customer → Order
g.add((EX.placesOrder, RDF.type, OWL.ObjectProperty))
g.add((EX.placesOrder, RDFS.domain, EX.Customer))
g.add((EX.placesOrder, RDFS.range, EX.Order))
g.add((EX.placesOrder, RDFS.label, Literal("places order")))

# containsProduct: Order → Product
g.add((EX.containsProduct, RDF.type, OWL.ObjectProperty))
g.add((EX.containsProduct, RDFS.domain, EX.Order))
g.add((EX.containsProduct, RDFS.range, EX.Product))
g.add((EX.containsProduct, RDFS.label, Literal("contains product")))

# belongsToCategory: Product → Category
g.add((EX.belongsToCategory, RDF.type, OWL.ObjectProperty))
g.add((EX.belongsToCategory, RDFS.domain, EX.Product))
g.add((EX.belongsToCategory, RDFS.range, EX.Category))

# --- Define Data Properties (literal values) ---
# hasPrice: Product → decimal
g.add((EX.hasPrice, RDF.type, OWL.DatatypeProperty))
g.add((EX.hasPrice, RDFS.domain, EX.Product))
g.add((EX.hasPrice, RDFS.range, XSD.decimal))

# orderDate: Order → date
g.add((EX.orderDate, RDF.type, OWL.DatatypeProperty))
g.add((EX.orderDate, RDFS.domain, EX.Order))
g.add((EX.orderDate, RDFS.range, XSD.date))

print("Ontology in Turtle format:\n")
print(g.serialize(format="turtle"))

In [ ]:
# Visualize the class hierarchy
G_onto = nx.DiGraph()

# Add class hierarchy
for s, p, o in g.triples((None, RDFS.subClassOf, None)):
    child = str(s).split("/")[-1]
    parent = str(o).split("/")[-1]
    G_onto.add_edge(child, parent, relation="subClassOf")

# Add property connections
for s, p, o in g.triples((None, RDFS.domain, None)):
    prop_name = str(s).split("/")[-1]
    for _, _, range_val in g.triples((s, RDFS.range, None)):
        domain = str(o).split("/")[-1]
        range_name = str(range_val).split("/")[-1].split("#")[-1]
        G_onto.add_edge(domain, range_name, relation=prop_name)

# Color classes vs datatypes
class_names = {str(s).split("/")[-1] for s, _, _ in g.triples((None, RDF.type, OWL.Class))}
color_map = ["#99CCFF" if n in class_names else "#FFCC99" for n in G_onto.nodes()]

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G_onto, seed=42, k=2)
nx.draw(G_onto, pos, with_labels=True, node_color=color_map,
        node_size=2500, font_size=9, font_weight="bold",
        arrows=True, arrowsize=15, edge_color="gray")

edge_labels = nx.get_edge_attributes(G_onto, "relation")
nx.draw_networkx_edge_labels(G_onto, pos, edge_labels=edge_labels, font_size=8)

legend_handles = [
    mpatches.Patch(color="#99CCFF", label="Classes"),
    mpatches.Patch(color="#FFCC99", label="Datatypes"),
]
plt.legend(handles=legend_handles, loc="upper left", fontsize=10)
plt.title("E-Commerce Ontology", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Competency Questions

**Competency questions** guide ontology design — they define what the ontology must be able to answer:

1. *"Which products did customer X order?"* → needs: Customer, Order, Product, placesOrder, containsProduct
2. *"What categories does product Y belong to?"* → needs: Product, Category, belongsToCategory
3. *"What is the price of product Z?"* → needs: Product, hasPrice

Let's populate the ontology with instances and test these questions with SPARQL.

In [ ]:
# Add instances (ABox / individuals)

# Categories
g.add((EX.Electronics, RDF.type, EX.Category))
g.add((EX.Electronics, RDFS.label, Literal("Electronics")))
g.add((EX.Computers, RDF.type, EX.Category))
g.add((EX.Computers, RDFS.label, Literal("Computers")))

# Products
g.add((EX.MacBookPro, RDF.type, EX.Laptop))
g.add((EX.MacBookPro, RDFS.label, Literal("MacBook Pro")))
g.add((EX.MacBookPro, EX.hasPrice, Literal(2499, datatype=XSD.decimal)))
g.add((EX.MacBookPro, EX.belongsToCategory, EX.Computers))
g.add((EX.MacBookPro, EX.belongsToCategory, EX.Electronics))

g.add((EX.iPhone, RDF.type, EX.ElectronicDevice))
g.add((EX.iPhone, RDFS.label, Literal("iPhone 15")))
g.add((EX.iPhone, EX.hasPrice, Literal(999, datatype=XSD.decimal)))
g.add((EX.iPhone, EX.belongsToCategory, EX.Electronics))

# Customers
g.add((EX.Alice, RDF.type, EX.Customer))
g.add((EX.Alice, RDFS.label, Literal("Alice")))
g.add((EX.Bob, RDF.type, EX.Customer))
g.add((EX.Bob, RDFS.label, Literal("Bob")))

# Orders
g.add((EX.Order001, RDF.type, EX.Order))
g.add((EX.Order001, EX.orderDate, Literal("2024-01-15", datatype=XSD.date)))
g.add((EX.Alice, EX.placesOrder, EX.Order001))
g.add((EX.Order001, EX.containsProduct, EX.MacBookPro))
g.add((EX.Order001, EX.containsProduct, EX.iPhone))

g.add((EX.Order002, RDF.type, EX.Order))
g.add((EX.Order002, EX.orderDate, Literal("2024-02-20", datatype=XSD.date)))
g.add((EX.Bob, EX.placesOrder, EX.Order002))
g.add((EX.Order002, EX.containsProduct, EX.iPhone))

print(f"Total triples in graph: {len(g)}")

In [ ]:
# CQ1: "Which products did Alice order?"
query1 = """
PREFIX ex: <http://example.org/ecommerce/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?product ?label ?price
WHERE {
    ex:Alice ex:placesOrder ?order .
    ?order ex:containsProduct ?product .
    ?product rdfs:label ?label .
    ?product ex:hasPrice ?price .
}
"""

print("CQ1: Which products did Alice order?\n")
for row in g.query(query1):
    print(f"  {row.label} (${row.price})")

In [ ]:
# CQ2: "What categories does MacBook Pro belong to?"
query2 = """
PREFIX ex: <http://example.org/ecommerce/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?category ?label
WHERE {
    ex:MacBookPro ex:belongsToCategory ?category .
    ?category rdfs:label ?label .
}
"""

print("CQ2: What categories does MacBook Pro belong to?\n")
for row in g.query(query2):
    print(f"  {row.label}")

In [ ]:
# CQ3: "What is the total value of Alice's orders?"
query3 = """
PREFIX ex: <http://example.org/ecommerce/>

SELECT (SUM(?price) AS ?total)
WHERE {
    ex:Alice ex:placesOrder ?order .
    ?order ex:containsProduct ?product .
    ?product ex:hasPrice ?price .
}
"""

print("CQ3: Total value of Alice's orders?\n")
for row in g.query(query3):
    print(f"  ${row.total}")

## 3. An Agentic AI Ontology

Now let's build a more relevant ontology for an **agentic AI system** with agents, tasks, tools, and capabilities.

In [ ]:
# Create a new graph for the agentic ontology
ag = Graph()
AGT = Namespace("http://example.org/agentic/")
ag.bind("agt", AGT)
ag.bind("owl", OWL)

# --- Classes ---
for cls in ["Agent", "Task", "Tool", "Document", "Capability", "DataType"]:
    ag.add((AGT[cls], RDF.type, OWL.Class))

# Subclasses of Agent
for sub in ["LLMAgent", "RetrievalAgent", "ValidationAgent"]:
    ag.add((AGT[sub], RDF.type, OWL.Class))
    ag.add((AGT[sub], RDFS.subClassOf, AGT.Agent))

# Subclasses of Task
for sub in ["ResearchTask", "SummarizationTask", "ValidationTask"]:
    ag.add((AGT[sub], RDF.type, OWL.Class))
    ag.add((AGT[sub], RDFS.subClassOf, AGT.Task))

# --- Properties ---
properties = [
    ("hasCapability", "Agent", "Capability"),
    ("requiresCapability", "Task", "Capability"),
    ("assignedTo", "Task", "Agent"),
    ("usesTool", "Agent", "Tool"),
    ("hasInput", "Task", "Document"),
    ("hasOutput", "Task", "Document"),
    ("requiresDataType", "Task", "DataType"),
    ("hasClearance", "Agent", "DataType"),
]

for prop, domain, range_ in properties:
    ag.add((AGT[prop], RDF.type, OWL.ObjectProperty))
    ag.add((AGT[prop], RDFS.domain, AGT[domain]))
    ag.add((AGT[prop], RDFS.range, AGT[range_]))

# Status property (data property)
ag.add((AGT.hasStatus, RDF.type, OWL.DatatypeProperty))
ag.add((AGT.hasStatus, RDFS.domain, AGT.Task))
ag.add((AGT.hasStatus, RDFS.range, XSD.string))

print("Agentic AI Ontology (Turtle):\n")
print(ag.serialize(format="turtle"))

In [ ]:
# Visualize the agentic ontology
G_ag = nx.DiGraph()

# Class hierarchy
for s, p, o in ag.triples((None, RDFS.subClassOf, None)):
    child = str(s).split("/")[-1]
    parent = str(o).split("/")[-1]
    G_ag.add_edge(child, parent, relation="subClassOf")

# Properties as edges between domain and range
for s, _, _ in ag.triples((None, RDF.type, OWL.ObjectProperty)):
    prop_name = str(s).split("/")[-1]
    for _, _, domain in ag.triples((s, RDFS.domain, None)):
        for _, _, range_ in ag.triples((s, RDFS.range, None)):
            d = str(domain).split("/")[-1]
            r = str(range_).split("/")[-1]
            G_ag.add_edge(d, r, relation=prop_name)

# Color by type
agent_classes = {"Agent", "LLMAgent", "RetrievalAgent", "ValidationAgent"}
task_classes = {"Task", "ResearchTask", "SummarizationTask", "ValidationTask"}
color_map = []
for n in G_ag.nodes():
    if n in agent_classes:
        color_map.append("#FF9999")
    elif n in task_classes:
        color_map.append("#99CCFF")
    else:
        color_map.append("#99FF99")

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G_ag, seed=42, k=2)
nx.draw(G_ag, pos, with_labels=True, node_color=color_map,
        node_size=2800, font_size=9, font_weight="bold",
        arrows=True, arrowsize=15, edge_color="gray")

edge_labels = nx.get_edge_attributes(G_ag, "relation")
nx.draw_networkx_edge_labels(G_ag, pos, edge_labels=edge_labels, font_size=8)

legend_handles = [
    mpatches.Patch(color="#FF9999", label="Agent classes"),
    mpatches.Patch(color="#99CCFF", label="Task classes"),
    mpatches.Patch(color="#99FF99", label="Other classes"),
]
plt.legend(handles=legend_handles, loc="upper left", fontsize=10)
plt.title("Agentic AI Ontology", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Populate with instances and query

# Capabilities
for cap in ["Summarization", "WebSearch", "CodeGeneration", "DataValidation"]:
    ag.add((AGT[cap], RDF.type, AGT.Capability))

# Data types
ag.add((AGT.PHI, RDF.type, AGT.DataType))
ag.add((AGT.PublicData, RDF.type, AGT.DataType))

# Agent instances
ag.add((AGT.AgentAlpha, RDF.type, AGT.LLMAgent))
ag.add((AGT.AgentAlpha, AGT.hasCapability, AGT.Summarization))
ag.add((AGT.AgentAlpha, AGT.hasCapability, AGT.CodeGeneration))
ag.add((AGT.AgentAlpha, AGT.hasClearance, AGT.PublicData))

ag.add((AGT.AgentBeta, RDF.type, AGT.RetrievalAgent))
ag.add((AGT.AgentBeta, AGT.hasCapability, AGT.WebSearch))
ag.add((AGT.AgentBeta, AGT.hasClearance, AGT.PublicData))
ag.add((AGT.AgentBeta, AGT.hasClearance, AGT.PHI))

ag.add((AGT.AgentGamma, RDF.type, AGT.ValidationAgent))
ag.add((AGT.AgentGamma, AGT.hasCapability, AGT.DataValidation))
ag.add((AGT.AgentGamma, AGT.hasClearance, AGT.PHI))

# Tasks
ag.add((AGT.Task1, RDF.type, AGT.SummarizationTask))
ag.add((AGT.Task1, AGT.requiresCapability, AGT.Summarization))
ag.add((AGT.Task1, AGT.hasStatus, Literal("pending")))

ag.add((AGT.Task2, RDF.type, AGT.ResearchTask))
ag.add((AGT.Task2, AGT.requiresCapability, AGT.WebSearch))
ag.add((AGT.Task2, AGT.requiresDataType, AGT.PHI))
ag.add((AGT.Task2, AGT.hasStatus, Literal("pending")))

print(f"Total triples: {len(ag)}")

In [ ]:
# Query: "Which agents can handle the summarization task?"
query_match = """
PREFIX agt: <http://example.org/agentic/>

SELECT ?agent ?capability
WHERE {
    agt:Task1 agt:requiresCapability ?capability .
    ?agent agt:hasCapability ?capability .
}
"""

print("Which agents can handle Task1 (Summarization)?\n")
for row in ag.query(query_match):
    agent = str(row.agent).split("/")[-1]
    cap = str(row.capability).split("/")[-1]
    print(f"  {agent} (has capability: {cap})")

In [ ]:
# Query: "Which agents have PHI clearance for Task2?"
query_clearance = """
PREFIX agt: <http://example.org/agentic/>

SELECT ?agent ?capability
WHERE {
    agt:Task2 agt:requiresCapability ?capability .
    agt:Task2 agt:requiresDataType ?datatype .
    ?agent agt:hasCapability ?capability .
    ?agent agt:hasClearance ?datatype .
}
"""

print("Which agents can handle Task2 (Research with PHI)?\n")
results = list(ag.query(query_clearance))
if results:
    for row in results:
        agent = str(row.agent).split("/")[-1]
        cap = str(row.capability).split("/")[-1]
        print(f"  {agent} (capability: {cap}, has PHI clearance)")
else:
    print("  No agents match all requirements.")

## 4. Common Pitfalls in Ontology Design

| Pitfall | Description | Example |
|---------|-------------|--------|
| **Missing annotations** | Classes without labels or descriptions | `ex:Cls1` with no `rdfs:label` |
| **Missing domain/range** | Properties without type constraints | `ex:relates` with no domain/range |
| **Orphaned classes** | Classes not connected to the hierarchy | A class with no subClassOf or instances |
| **Over-modeling** | Too many classes for edge cases | Separate classes for every product variant |
| **Inconsistencies** | Contradictory axioms | `A subClassOf B` and `B subClassOf A` (cycle) |
| **Misuse of inheritance** | Using subClassOf when composition is better | `RedCar subClassOf Car` vs `Car hasColor Red` |

## Key Takeaways

- Ontologies provide the **semantic backbone** for knowledge graphs — defining what's allowed
- **Competency questions** drive design decisions: every class and property should serve a purpose
- In agentic systems, ontologies enable **type-safe task routing**: matching agent capabilities to task requirements, enforcing data access policies
- Start simple, iterate, and align with existing standards where possible
- LLMs can help draft ontologies, but expert validation and reasoning checks are essential